# Handmade Products Recommendation – Part 2: Sparsity Simulation, Models & Evaluation
**Continues from 01_data_preprocessing.ipynb** — assumes `ml_clean` and `hm_clean` are already built.
Run `notebooks/01_data_preprocessing.ipynb` first, then run this notebook.

In [ ]:
from pathlib import Path
# =============================================================
# IMPORTS
# =============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import json, os, time, warnings, tracemalloc
warnings.filterwarnings('ignore')

from scipy.sparse import csr_matrix
from scipy.sparse.linalg import svds
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MultiLabelBinarizer

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
os.makedirs('plots', exist_ok=True)
os.makedirs('results', exist_ok=True)

PALETTE = ["#2E75B6", "#FFC000", "#70AD47", "#FF5252"]
plt.rcParams.update({'figure.dpi': 150, 'savefig.bbox': 'tight'})
print('Imports OK')

In [ ]:
# =============================================================
# RE-LOAD DATA  (load saved clean datasets if available)
# so this notebook can use the outputs from 01_data_preprocessing.ipynb
# =============================================================
THRESHOLD = 4
MIN_INTERACTIONS = 2

saved_dir = 'saved'
saved_ml_clean = os.path.join(saved_dir, 'ml_clean.pkl')
saved_ml_movies = os.path.join(saved_dir, 'ml_movies.pkl')
saved_hm_clean = os.path.join(saved_dir, 'hm_clean.pkl')

if os.path.exists(saved_ml_clean) and os.path.exists(saved_ml_movies) and os.path.exists(saved_hm_clean):
    ml_clean = pd.read_pickle(saved_ml_clean)
    ml_movies = pd.read_pickle(saved_ml_movies)
    hm_clean = pd.read_pickle(saved_hm_clean)
    print('Loaded saved clean datasets from saved/.')
    print(f'  ml_clean: {len(ml_clean):,} rows')
    print(f'  ml_movies: {len(ml_movies):,} rows')
    print(f'  hm_clean: {len(hm_clean):,} rows')
else:
    print('Saved clean datasets not found; rebuilding from raw files.')
    ml_ratings = pd.read_csv(str(Path('data/raw/ratings.dat')), sep='::', header=None,
        names=['UserID','MovieID','Rating','Timestamp'], engine='python')
    ml_movies  = pd.read_csv(str(Path('data/raw/movies.dat')), sep='::', header=None,
        names=['MovieID','Title','Genres'], engine='python', encoding='latin-1')

    ml_clean = ml_ratings[ml_ratings['Rating'] >= THRESHOLD].copy()
    ml_clean = ml_clean.drop_duplicates(subset=['UserID','MovieID'])
    ml_clean['UserID_idx'] = ml_clean['UserID'].astype('category').cat.codes
    ml_clean['ItemID_idx'] = ml_clean['MovieID'].astype('category').cat.codes

    records = []
    with open(str(Path('data/raw/Handmade_Products.jsonl')), 'r') as f:
        for line in f:
            d = json.loads(line.strip())
            records.append({'UserID': d.get('user_id'), 'ItemID': d.get('asin'),
                            'Rating': d.get('rating'), 'Timestamp': d.get('timestamp')})
    hm_raw = pd.DataFrame(records)
    hm = hm_raw.dropna(subset=['UserID','ItemID','Rating']).copy()
    hm = hm.drop_duplicates(subset=['UserID','ItemID'])
    hm = hm[(hm['Rating'] >= 1) & (hm['Rating'] <= 5)]
    user_counts = hm.groupby('UserID').size()
    hm = hm[hm['UserID'].isin(user_counts[user_counts >= MIN_INTERACTIONS].index)]
    hm_clean = hm[hm['Rating'] >= THRESHOLD].copy()
    hm_clean['UserID_idx'] = hm_clean['UserID'].astype('category').cat.codes
    hm_clean['ItemID_idx'] = hm_clean['ItemID'].astype('category').cat.codes
    print('Rebuilt clean datasets from raw files.')


---
## 4.2  Sparsity Simulation

In [ ]:
# =============================================================
# 4.2  SPARSITY SIMULATION — MovieLens at 4 density levels
# =============================================================
print('=' * 60)
print('4.2  SPARSITY SIMULATION')
print('=' * 60)

DENSITY_LEVELS = [1.0, 0.5, 0.25, 0.10]   # 100%, 50%, 25%, 10%
LEVEL_LABELS   = ['100%', '50%', '25%', '10%']

def sample_sparse(df, fraction, user_col='UserID_idx', seed=SEED):
    """Randomly sample `fraction` of interactions.
    Guarantees each user keeps at least 1 interaction."""
    rng = np.random.default_rng(seed)
    sampled_parts = []
    for uid, group in df.groupby(user_col):
        n_keep = max(1, int(round(len(group) * fraction)))
        idx = rng.choice(len(group), size=n_keep, replace=False)
        sampled_parts.append(group.iloc[idx])
    return pd.concat(sampled_parts).reset_index(drop=True)

ml_sparse = {}
for frac, label in zip(DENSITY_LEVELS, LEVEL_LABELS):
    df_s = sample_sparse(ml_clean, frac)
    nu   = df_s['UserID_idx'].nunique()
    ni   = df_s['ItemID_idx'].nunique()
    d    = len(df_s) / (nu * ni)
    ml_sparse[label] = df_s
    print(f'  Level {label:>5}: {len(df_s):>7,} interactions  |  '
          f'{nu:,} users  |  {ni:,} items  |  density {d:.4%}')

print('\n  Sparsity simulation complete.')

---
## 4.4  Train / Test Split (temporal)

In [ ]:
# =============================================================
# 4.4  TEMPORAL TRAIN / TEST SPLIT
# =============================================================
print('=' * 60)
print('4.4  TEMPORAL TRAIN/TEST SPLIT')
print('=' * 60)

def temporal_split(df, test_frac=0.20,
                   user_col='UserID_idx', item_col='ItemID_idx',
                   time_col='Timestamp'):
    """For each user put their most recent `test_frac` interactions
    in the test set; the rest go to train."""
    train_parts, test_parts = [], []
    for uid, grp in df.groupby(user_col):
        grp_sorted = grp.sort_values(time_col)
        n_test = max(1, int(len(grp_sorted) * test_frac))
        test_parts.append(grp_sorted.tail(n_test))
        train_parts.append(grp_sorted.head(len(grp_sorted) - n_test))
    train = pd.concat(train_parts).reset_index(drop=True)
    test  = pd.concat(test_parts).reset_index(drop=True)
    return train, test

# ── Build splits for each sparsity level ─────────────────────
splits = {}
for label, df_s in ml_sparse.items():
    train, test = temporal_split(df_s)
    splits[label] = {'train': train, 'test': test}
    print(f'  Level {label:>5}: train={len(train):,}  test={len(test):,}')

# ── Handmade Products split ───────────────────────────────────
hm_train, hm_test = temporal_split(hm_clean)
print(f'  Handmade:    train={len(hm_train):,}  test={len(hm_test):,}')

---
## Evaluation helpers (Precision@K, Recall@K, NDCG@K)

In [ ]:
# =============================================================
# EVALUATION METRICS
# =============================================================
def precision_at_k(recommended, relevant, k):
    recs = recommended[:k]
    hits = len(set(recs) & set(relevant))
    return hits / k

def recall_at_k(recommended, relevant, k):
    if not relevant:
        return 0.0
    recs = recommended[:k]
    hits = len(set(recs) & set(relevant))
    return hits / len(relevant)

def ndcg_at_k(recommended, relevant, k):
    recs = recommended[:k]
    rel_set = set(relevant)
    dcg  = sum(1.0 / np.log2(i + 2)
               for i, r in enumerate(recs) if r in rel_set)
    idcg = sum(1.0 / np.log2(i + 2)
               for i in range(min(len(relevant), k)))
    return dcg / idcg if idcg > 0 else 0.0

def evaluate_model(get_recs_fn, test_df, train_df,
                   user_col='UserID_idx', item_col='ItemID_idx',
                   K=10):
    """
    get_recs_fn(user_id, trained_items) -> list of recommended item ids (ranked)
    Returns dict with mean Precision, Recall, NDCG and inference latency.
    """
    # Build ground-truth: test items per user
    gt = test_df.groupby(user_col)[item_col].apply(list).to_dict()
    # Build training items per user (to exclude from recs)
    trained = train_df.groupby(user_col)[item_col].apply(set).to_dict()

    precs, recs, ndcgs = [], [], []
    latencies = []

    for uid, relevant in gt.items():
        t0 = time.perf_counter()
        recommended = get_recs_fn(uid, trained.get(uid, set()))
        latencies.append(time.perf_counter() - t0)

        precs.append(precision_at_k(recommended, relevant, K))
        recs.append(recall_at_k(recommended, relevant, K))
        ndcgs.append(ndcg_at_k(recommended, relevant, K))

    return {
        f'Precision@{K}': np.mean(precs),
        f'Recall@{K}':    np.mean(recs),
        f'NDCG@{K}':      np.mean(ndcgs),
        'Latency_ms':     np.mean(latencies) * 1000
    }

print('Evaluation helpers defined.')

---
## 4.3  Model 1 – Content-Based Filtering

In [ ]:
# =============================================================
# MODEL 1: CONTENT-BASED FILTERING (genre cosine similarity)
# =============================================================
print('=' * 60)
print('MODEL 1: CONTENT-BASED FILTERING')
print('=' * 60)

# Build genre feature matrix for MovieLens
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(
    ml_movies['Genres'].str.split('|')
)
genre_df = pd.DataFrame(genre_matrix,
                        index=ml_movies['MovieID'],
                        columns=mlb.classes_)

# Precompute item-item cosine similarity
item_sim = cosine_similarity(genre_matrix)
movieid_to_row = {mid: i for i, mid in enumerate(ml_movies['MovieID'])}
row_to_movieid = {i: mid for mid, i in movieid_to_row.items()}

# We also need a mapping from ItemID_idx (used in ml_clean) to MovieID
idx_to_movieid = dict(zip(ml_clean['ItemID_idx'], ml_clean['MovieID']))
movieid_to_idx = {v: k for k, v in idx_to_movieid.items()}

def cb_get_recs(uid, trained_items, n=50):
    """For a user, aggregate similarity scores from their liked items,
    return top-n unseen item indices."""
    # Get MovieIDs this user has interacted with (in training set)
    user_movieids = [idx_to_movieid[i] for i in trained_items
                     if i in idx_to_movieid]
    if not user_movieids:
        return []  # cold-start: no history

    # Sum similarity scores across all liked items
    scores = np.zeros(len(ml_movies))
    for mid in user_movieids:
        if mid in movieid_to_row:
            scores += item_sim[movieid_to_row[mid]]

    # Zero out already-seen items
    for mid in user_movieids:
        if mid in movieid_to_row:
            scores[movieid_to_row[mid]] = 0

    # Return top-n as ItemID_idx values
    top_rows = np.argsort(scores)[::-1][:n]
    top_movieids = [row_to_movieid[r] for r in top_rows]
    return [movieid_to_idx[m] for m in top_movieids if m in movieid_to_idx]

print('Content-Based model built.')

---
## 4.3  Model 2 – Matrix Factorization (SVD)

In [ ]:
# =============================================================
# MODEL 2: MATRIX FACTORIZATION (SVD with implicit feedback)
# =============================================================
print('=' * 60)
print('MODEL 2: MATRIX FACTORIZATION (SVD)')
print('=' * 60)

class MFModel:
    def __init__(self, n_factors=50):
        self.n_factors = n_factors
        self.U = None   # user embeddings
        self.V = None   # item embeddings
        self.n_users = None
        self.n_items = None

    def fit(self, train_df, user_col='UserID_idx', item_col='ItemID_idx'):
        self.n_users = train_df[user_col].max() + 1
        self.n_items = train_df[item_col].max() + 1
        # Build sparse user-item matrix (binary implicit)
        row = train_df[user_col].values
        col = train_df[item_col].values
        data = np.ones(len(train_df))
        R = csr_matrix((data, (row, col)),
                       shape=(self.n_users, self.n_items))
        # Truncated SVD
        k = min(self.n_factors, min(R.shape) - 1)
        U, s, Vt = svds(R.astype(float), k=k)
        # Absorb singular values into U
        self.U = U * s
        self.V = Vt.T

    def predict_user(self, uid):
        """Return score vector across all items for user uid."""
        if uid >= self.n_users:
            return np.zeros(self.n_items)
        return self.U[uid] @ self.V.T

    def get_recs(self, uid, trained_items, n=50):
        scores = self.predict_user(uid).copy()
        for i in trained_items:
            if i < len(scores):
                scores[i] = -np.inf
        return list(np.argsort(scores)[::-1][:n])

print('MF class defined.')

---
## 4.3  Model 3 – Neural Collaborative Filtering (NCF)

In [ ]:
# =============================================================
# MODEL 3: NEURAL COLLABORATIVE FILTERING (NCF)
# =============================================================
print('=' * 60)
print('MODEL 3: NEURAL COLLABORATIVE FILTERING')
print('=' * 60)

class InteractionDataset(Dataset):
    """Yields (user, pos_item, neg_item) triples for BPR-style training."""
    def __init__(self, train_df, n_items,
                 user_col='UserID_idx', item_col='ItemID_idx',
                 n_neg=4):
        self.n_items = n_items
        self.n_neg   = n_neg
        self.pos_pairs = list(zip(train_df[user_col], train_df[item_col]))
        self.user_pos  = (
            train_df.groupby(user_col)[item_col]
                    .apply(set).to_dict()
        )

    def __len__(self):
        return len(self.pos_pairs) * self.n_neg

    def __getitem__(self, idx):
        u, pos = self.pos_pairs[idx % len(self.pos_pairs)]
        pos_set = self.user_pos.get(u, set())
        neg = np.random.randint(0, self.n_items)
        while neg in pos_set:
            neg = np.random.randint(0, self.n_items)
        return torch.tensor(u, dtype=torch.long), \
               torch.tensor(pos, dtype=torch.long), \
               torch.tensor(neg, dtype=torch.long)


class NCF(nn.Module):
    def __init__(self, n_users, n_items, emb_dim=32, layers=[64, 32, 16]):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        # MLP layers
        mlp_input = emb_dim * 2
        mlp_layers = []
        for hidden in layers:
            mlp_layers += [nn.Linear(mlp_input, hidden),
                           nn.ReLU(),
                           nn.Dropout(0.2)]
            mlp_input = hidden
        self.mlp     = nn.Sequential(*mlp_layers)
        self.out     = nn.Linear(mlp_input, 1)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)

    def forward(self, users, items):
        u = self.user_emb(users)
        i = self.item_emb(items)
        x = torch.cat([u, i], dim=-1)
        return self.out(self.mlp(x)).squeeze(-1)


class NCFModel:
    def __init__(self, emb_dim=32, layers=[64, 32, 16],
                 lr=1e-3, epochs=10, batch_size=512, n_neg=4):
        self.emb_dim    = emb_dim
        self.layers     = layers
        self.lr         = lr
        self.epochs     = epochs
        self.batch_size = batch_size
        self.n_neg      = n_neg
        self.model      = None
        self.n_items    = None

    def fit(self, train_df, user_col='UserID_idx', item_col='ItemID_idx'):
        n_users = train_df[user_col].max() + 1
        self.n_items = train_df[item_col].max() + 1

        dataset = InteractionDataset(train_df, self.n_items,
                                     user_col, item_col, self.n_neg)
        loader  = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)

        self.model = NCF(n_users, self.n_items, self.emb_dim, self.layers)
        opt = optim.Adam(self.model.parameters(), lr=self.lr)
        loss_fn = nn.BCEWithLogitsLoss()

        self.model.train()
        for epoch in range(self.epochs):
            total_loss = 0
            for u, pos, neg in loader:
                pos_score = self.model(u, pos)
                neg_score = self.model(u, neg)
                scores = torch.cat([pos_score, neg_score])
                labels = torch.cat([torch.ones(len(pos_score)),
                                    torch.zeros(len(neg_score))])
                loss = loss_fn(scores, labels)
                opt.zero_grad()
                loss.backward()
                opt.step()
                total_loss += loss.item()
            if (epoch + 1) % 5 == 0:
                print(f'    Epoch {epoch+1}/{self.epochs}  '
                      f'loss={total_loss/len(loader):.4f}')

    def get_recs(self, uid, trained_items, n=50):
        if self.model is None:
            return []
        self.model.eval()
        with torch.no_grad():
            u_tensor = torch.tensor([uid] * self.n_items, dtype=torch.long)
            i_tensor = torch.arange(self.n_items, dtype=torch.long)
            scores   = self.model(u_tensor, i_tensor).numpy()
        for i in trained_items:
            if i < self.n_items:
                scores[i] = -np.inf
        return list(np.argsort(scores)[::-1][:n])

print('NCF class defined.')

---
## 4.3 + 6  Train all models across sparsity levels & collect results

In [ ]:
# =============================================================
# MAIN EXPERIMENT LOOP
# MovieLens — 4 sparsity levels × 3 models
# =============================================================
print('=' * 60)
print('MAIN EXPERIMENT — MovieLens sparsity sweep')
print('=' * 60)

K = 10
all_results = []  # will hold one dict per (level, model)

for label in LEVEL_LABELS:
    train = splits[label]['train']
    test  = splits[label]['test']
    print(f'\n── Level {label} ──────────────────────────────')

    # ── 1. Content-Based ────────────────────────────────────
    print('  [CB] evaluating...')
    t0 = time.perf_counter()
    tracemalloc.start()
    # CB is not trained (precomputed item similarity matrix)
    cb_metrics = evaluate_model(cb_get_recs, test, train, K=K)
    _, cb_mem_peak = tracemalloc.stop(), tracemalloc.get_traced_memory()[1]
    tracemalloc.stop()
    cb_train_time = time.perf_counter() - t0
    cb_metrics.update({'Model': 'Content-Based', 'Level': label,
                       'Train_time_s': cb_train_time, 'Memory_MB': 0})
    all_results.append(cb_metrics)
    print(f"    P@{K}={cb_metrics[f'Precision@{K}']:.4f}  "
          f"R@{K}={cb_metrics[f'Recall@{K}']:.4f}  "
          f"NDCG@{K}={cb_metrics[f'NDCG@{K}']:.4f}")

    # ── 2. Matrix Factorization ──────────────────────────────
    print('  [MF] training...')
    tracemalloc.start()
    t0 = time.perf_counter()
    mf = MFModel(n_factors=50)
    mf.fit(train)
    mf_train_time = time.perf_counter() - t0
    _, mf_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    mf_metrics = evaluate_model(
        lambda uid, ti: mf.get_recs(uid, ti, n=K),
        test, train, K=K)
    mf_metrics.update({'Model': 'Matrix Factorization', 'Level': label,
                       'Train_time_s': mf_train_time,
                       'Memory_MB': mf_mem / 1e6})
    all_results.append(mf_metrics)
    print(f"    P@{K}={mf_metrics[f'Precision@{K}']:.4f}  "
          f"R@{K}={mf_metrics[f'Recall@{K}']:.4f}  "
          f"NDCG@{K}={mf_metrics[f'NDCG@{K}']:.4f}  "
          f"train={mf_train_time:.1f}s")

    # ── 3. NCF ───────────────────────────────────────────────
    print('  [NCF] training (10 epochs)...')
    tracemalloc.start()
    t0 = time.perf_counter()
    ncf = NCFModel(emb_dim=32, layers=[64, 32, 16],
                   epochs=10, batch_size=512)
    ncf.fit(train)
    ncf_train_time = time.perf_counter() - t0
    _, ncf_mem = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    ncf_metrics = evaluate_model(
        lambda uid, ti: ncf.get_recs(uid, ti, n=K),
        test, train, K=K)
    ncf_metrics.update({'Model': 'NCF', 'Level': label,
                        'Train_time_s': ncf_train_time,
                        'Memory_MB': ncf_mem / 1e6})
    all_results.append(ncf_metrics)
    print(f"    P@{K}={ncf_metrics[f'Precision@{K}']:.4f}  "
          f"R@{K}={ncf_metrics[f'Recall@{K}']:.4f}  "
          f"NDCG@{K}={ncf_metrics[f'NDCG@{K}']:.4f}  "
          f"train={ncf_train_time:.1f}s")

results_df = pd.DataFrame(all_results)
results_df.to_csv('results/ml_results.csv', index=False)
print('\nResults saved to results/ml_results.csv')
print(results_df.to_string())

In [ ]:
# =============================================================
# HANDMADE PRODUCTS — single extreme sparsity evaluation
# =============================================================
print('=' * 60)
print('HANDMADE PRODUCTS — extreme sparsity evaluation')
print('=' * 60)

hm_results = []

# ── MF on Handmade ──────────────────────────────────────────
print('  [MF] training on Handmade...')
t0 = time.perf_counter()
mf_hm = MFModel(n_factors=50)
mf_hm.fit(hm_train)
hm_mf_metrics = evaluate_model(
    lambda uid, ti: mf_hm.get_recs(uid, ti, n=K),
    hm_test, hm_train, K=K)
hm_mf_metrics.update({'Model': 'Matrix Factorization',
                      'Level': 'Handmade',
                      'Train_time_s': time.perf_counter() - t0})
hm_results.append(hm_mf_metrics)

# ── NCF on Handmade ─────────────────────────────────────────
print('  [NCF] training on Handmade (10 epochs)...')
t0 = time.perf_counter()
ncf_hm = NCFModel(emb_dim=32, layers=[64, 32, 16], epochs=10)
ncf_hm.fit(hm_train)
hm_ncf_metrics = evaluate_model(
    lambda uid, ti: ncf_hm.get_recs(uid, ti, n=K),
    hm_test, hm_train, K=K)
hm_ncf_metrics.update({'Model': 'NCF',
                       'Level': 'Handmade',
                       'Train_time_s': time.perf_counter() - t0})
hm_results.append(hm_ncf_metrics)

hm_results_df = pd.DataFrame(hm_results)
hm_results_df.to_csv('results/hm_results.csv', index=False)
print('Handmade results:')
print(hm_results_df.to_string())

---
## Cold-Start Evaluation

In [ ]:
# =============================================================
# COLD-START EVALUATION (users with ≤ 5 training interactions)
# =============================================================
print('=' * 60)
print('COLD-START EVALUATION')
print('=' * 60)

# Use the full MovieLens split (100% level)
train_full = splits['100%']['train']
test_full  = splits['100%']['test']

# Identify cold-start users: <= 5 training interactions
user_train_counts = train_full.groupby('UserID_idx').size()
cold_users = set(user_train_counts[user_train_counts <= 5].index)
cold_test  = test_full[test_full['UserID_idx'].isin(cold_users)]
print(f'  Cold-start users: {len(cold_users):,}'
      f'  ({len(cold_users)/train_full["UserID_idx"].nunique()*100:.1f}% of users)')
print(f'  Cold-start test interactions: {len(cold_test):,}')

cold_results = []

# CB on cold-start
cb_cold = evaluate_model(cb_get_recs, cold_test, train_full, K=K)
cb_cold['Model'] = 'Content-Based'
cold_results.append(cb_cold)

# MF on cold-start (reuse mf trained on 100%)
mf_full = MFModel(n_factors=50)
mf_full.fit(train_full)
mf_cold = evaluate_model(
    lambda uid, ti: mf_full.get_recs(uid, ti, n=K),
    cold_test, train_full, K=K)
mf_cold['Model'] = 'Matrix Factorization'
cold_results.append(mf_cold)

# NCF on cold-start
ncf_full = NCFModel(emb_dim=32, layers=[64, 32, 16], epochs=10)
ncf_full.fit(train_full)
ncf_cold = evaluate_model(
    lambda uid, ti: ncf_full.get_recs(uid, ti, n=K),
    cold_test, train_full, K=K)
ncf_cold['Model'] = 'NCF'
cold_results.append(ncf_cold)

cold_df = pd.DataFrame(cold_results)
cold_df.to_csv('results/cold_start_results.csv', index=False)
print('\nCold-start results:')
print(cold_df[['Model', f'Precision@{K}', f'Recall@{K}', f'NDCG@{K}']].to_string())

---
## Section 6 – Results Plots

In [ ]:
# =============================================================
# PLOT 8: Degradation Curves — Precision, Recall, NDCG vs Sparsity
# =============================================================
print('Generating results plots...')

models_order  = ['Content-Based', 'Matrix Factorization', 'NCF']
model_colors  = {'Content-Based': PALETTE[2],
                 'Matrix Factorization': PALETTE[0],
                 'NCF': PALETTE[3]}
model_markers = {'Content-Based': 's',
                 'Matrix Factorization': 'o',
                 'NCF': '^'}
x_labels = ['100%', '50%', '25%', '10%']

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
fig.suptitle('Figure 8 – Ranking Performance Degradation Curves (MovieLens 1M)',
             fontsize=13, fontweight='bold')

metrics = [f'Precision@{K}', f'Recall@{K}', f'NDCG@{K}']
for ax, metric in zip(axes, metrics):
    for model in models_order:
        sub = results_df[results_df['Model'] == model]
        sub = sub.set_index('Level').reindex(x_labels)
        ax.plot(x_labels, sub[metric].values,
                marker=model_markers[model],
                color=model_colors[model],
                linewidth=2, markersize=7, label=model)
    ax.set_title(metric, fontsize=11)
    ax.set_xlabel('Interaction Coverage')
    ax.set_ylabel(metric)
    ax.legend(fontsize=8)
    ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('plots/fig8_degradation_curves.png')
plt.close()
print('  ✓ Saved: plots/fig8_degradation_curves.png')

In [ ]:
# =============================================================
# PLOT 9: Model Comparison — full dataset summary bar chart
# =============================================================
full_results = results_df[results_df['Level'] == '100%'].copy()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
fig.suptitle('Figure 9 – Model Comparison at Full Interaction Coverage (100%)',
             fontsize=13, fontweight='bold')

for ax, metric in zip(axes, metrics):
    bars = ax.bar(full_results['Model'], full_results[metric],
                  color=[model_colors[m] for m in full_results['Model']],
                  edgecolor='white', width=0.5)
    ax.set_title(metric, fontsize=11)
    ax.set_ylabel(metric)
    ax.set_ylim(0, full_results[metric].max() * 1.25)
    for bar, val in zip(bars, full_results[metric]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.001, f'{val:.4f}',
                ha='center', fontsize=9)
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('plots/fig9_model_comparison.png')
plt.close()
print('  ✓ Saved: plots/fig9_model_comparison.png')

In [ ]:
# =============================================================
# PLOT 10: Training Time vs Sparsity Level
# =============================================================
fig, ax = plt.subplots(figsize=(9, 4))
fig.suptitle('Figure 10 – Training Time vs Interaction Coverage',
             fontsize=13, fontweight='bold')

for model in ['Matrix Factorization', 'NCF']:
    sub = results_df[results_df['Model'] == model]
    sub = sub.set_index('Level').reindex(x_labels)
    ax.plot(x_labels, sub['Train_time_s'].values,
            marker=model_markers[model],
            color=model_colors[model],
            linewidth=2, markersize=7, label=model)

ax.set_xlabel('Interaction Coverage')
ax.set_ylabel('Training Time (seconds)')
ax.legend(fontsize=9)
ax.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.savefig('plots/fig10_training_time.png')
plt.close()
print('  ✓ Saved: plots/fig10_training_time.png')

In [ ]:
# =============================================================
# PLOT 11: Cold-Start Performance Bar Chart
# =============================================================
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
fig.suptitle('Figure 11 – Cold-Start Performance (≤5 training interactions per user)',
             fontsize=12, fontweight='bold')

for ax, metric in zip(axes, metrics):
    bars = ax.bar(cold_df['Model'], cold_df[metric],
                  color=[model_colors[m] for m in cold_df['Model']],
                  edgecolor='white', width=0.5)
    ax.set_title(metric, fontsize=11)
    ax.set_ylabel(metric)
    ax.set_ylim(0, max(cold_df[metric].max() * 1.3, 0.01))
    for bar, val in zip(bars, cold_df[metric]):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.0005, f'{val:.4f}',
                ha='center', fontsize=9)
    ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig('plots/fig11_cold_start.png')
plt.close()
print('  ✓ Saved: plots/fig11_cold_start.png')

In [ ]:
# =============================================================
# SUMMARY TABLES for thesis Section 6
# =============================================================
print('\n' + '=' * 60)
print('SUMMARY TABLES FOR SECTION 6')
print('=' * 60)

print('\n── Table 3: Full results across all sparsity levels ──')
table3 = results_df[['Model', 'Level',
                      f'Precision@{K}', f'Recall@{K}', f'NDCG@{K}',
                      'Train_time_s', 'Latency_ms']].copy()
for col in [f'Precision@{K}', f'Recall@{K}', f'NDCG@{K}']:
    table3[col] = table3[col].round(4)
table3['Train_time_s'] = table3['Train_time_s'].round(2)
table3['Latency_ms']   = table3['Latency_ms'].round(3)
print(table3.to_string(index=False))

print('\n── Table 4: Cold-start results ──')
table4 = cold_df[['Model', f'Precision@{K}', f'Recall@{K}', f'NDCG@{K}']].copy()
for col in [f'Precision@{K}', f'Recall@{K}', f'NDCG@{K}']:
    table4[col] = table4[col].round(4)
print(table4.to_string(index=False))

print('\n── Table 5: Handmade Products (extreme sparsity) ──')
table5 = hm_results_df[['Model', f'Precision@{K}', f'Recall@{K}', f'NDCG@{K}',
                          'Train_time_s']].copy()
for col in [f'Precision@{K}', f'Recall@{K}', f'NDCG@{K}']:
    table5[col] = table5[col].round(4)
table5['Train_time_s'] = table5['Train_time_s'].round(2)
print(table5.to_string(index=False))

print('\n✓ All done! Plots in /plots, CSVs in /results')